In [99]:
# ============================================================
# Project : E-Commerce Orders — Data Cleaning Practice
# Purpose : Reinforce all 7 phases on a new business domain
# Input   : In-memory DataFrame (15 rows × 14 columns)
# Output  : Formatted Excel workbook + automated email
# Run     : Execute cells top to bottom in Jupyter
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ── All path and parameter constants at the top ───────────────────────
DATA_PATH       = Path("data/ecommerce_orders.csv")
OUTPUT_FOLDER   = Path("output/")
EXPECTED_COLS   = [
    "order_id", "customer_id", "customer_name", "email",
    "product_category", "product_name", "quantity", "unit_price",
    "discount_pct", "order_date", "delivery_date",
    "status", "payment_method", "country"
]

np.random.seed(7)   # different seed = different dataset to the sales notebook

# ── Build the messy dataset ───────────────────────────────────────────
data = {
    "order_id": [1001,1002,1003,1004,1005,1006,1007,
                 1008,1009,1010,1011,1002,1012,1013,1014],

    "customer_id": ["C001","c002","C003","C004","c005","C006","C007",
                    "C008","c009","C010","C011","c002","C012","C013","C014"],

    "customer_name": ["Alice Johnson","  bob smith","CHARLIE BROWN",
                      "Diana Prince",None,"EVE ADAMS","Frank Castle",
                      "  grace hopper","HENRY FORD","ida lupino",
                      "Jack Ryan","  bob smith","Karen Page",
                      "LUKE CAGE","  maya gold  "],

    "email": ["alice@shop.com","bob@shop.com","charlie_noemail",
              "diana@shop.com","eve@shop.com","notanemail","frank@shop.com",
              "grace@shop.com","henry@shop.com","ida@shop.com",
              "jack@shop.com","bob@shop.com","karen@shop.com",
              "luke@shop.com","maya@shop.com"],

    "product_category": ["Electronics","clothing","ELECTRONICS",
                         "Clothing","food & drink","Food & Drink",
                         "electronics","CLOTHING","Electronics",
                         "food & drink","Clothing","clothing",
                         "ELECTRONICS","Food & Drink","Electronics"],

    "product_name": ["Laptop Pro  ","cotton t-shirt","LAPTOP PRO",
                     "  Denim Jeans","Organic Coffee","ORGANIC COFFEE",
                     "Wireless Mouse","Wool Sweater  ","USB-C Hub",
                     "Green Tea Pack","Slim Trousers","cotton t-shirt",
                     "  Laptop Stand","Sparkling Water","Bluetooth Speaker"],

    "quantity": [1, 3, 1, 2, 5, None, 2, 1, 4, 6, 2, 3, 1, -3, 2],

    # NOTE: unit_price has "$" prefix on some rows — will be stored as object dtype
    "unit_price": ["$1200.00", "$25.00", "$1200.00", "$55.00",
                   "$12.50", "$12.50", "$35.00", "$80.00",
                   "$45.00", "$8.00", "$65.00", "$25.00",
                   None, "$3.50", "$95.00"],

    # NOTE: discount_pct — some rows use 0–1 scale, others use 0–100 scale (BUG)
    "discount_pct": [0.10, 0.05, 10.0, 0.15, 0.0, 5.0,
                     0.20, 0.0, 0.10, 0.05, 20.0, 0.05,
                     0.10, 0.0, 0.08],

    # NOTE: mixed date formats — some DD/MM/YYYY, some YYYY-MM-DD
    "order_date": ["2024-01-10","15/01/2024","2024-01-22",
                   "28/01/2024","2024-02-03","2024-02-10",
                   "14/02/2024","2024-02-20","2024-03-01",
                   "05/03/2024","2024-03-15","15/01/2024",
                   "2024-03-22","28/03/2024","2024-04-05"],

    # NOTE: one delivery_date is BEFORE the order_date — logical violation
    "delivery_date": ["2024-01-14","2024-01-19","2024-01-10",
                      "2024-02-04","2024-02-10","2024-02-18",
                      "2024-02-21","2024-02-28","2024-03-09",
                      "2024-03-14","2024-03-22","2024-01-19",
                      "2024-03-30","2024-04-06","2024-04-14"],

    "status": ["Delivered","Shipped","delivered","DELIVERED",
               "Processing","shipped","Delivered","SHIPPED",
               "Delivered","Processing","delivered","Shipped",
               "Delivered","delivered","Processing"],

    "payment_method": ["Credit Card","PayPal","Credit Card",None,
                       "PayPal","Credit Card","Debit Card","PayPal",
                       "Credit Card","Debit Card",None,"PayPal",
                       "Credit Card","Debit Card","PayPal"],

    "country": ["USA","UK","United States","GB","France",
                "FR","USA","United Kingdom","US","France",
                "UK","UK","United States","FR","USA"],
}

df = pd.DataFrame(data)

# ── Save to CSV so the pipeline can load it from disk ────────────────
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PATH, index=False)

print(f"[DATASET] {df.shape[0]} rows × {df.shape[1]} columns created")
print(f"[SAVED]   {DATA_PATH.resolve()}")
df

[DATASET] 15 rows × 14 columns created
[SAVED]   C:\Users\aman9\OneDrive\Desktop\My code\data\ecommerce_orders.csv


,order_id,customer_id,customer_name,email,product_category,product_name,quantity,unit_price,discount_pct,order_date,delivery_date,status,payment_method,country
0,1001,C001,Alice Johnson,alice@shop.com,Electronics,Laptop Pro,1.0,$1200.00,0.10,2024-01-10,2024-01-14,Delivered,Credit Card,USA
1,1002,c002,bob smith,bob@shop.com,clothing,cotton t-shirt,3.0,$25.00,0.05,15/01/2024,2024-01-19,Shipped,PayPal,UK
2,1003,C003,CHARLIE BROWN,charlie_noemail,ELECTRONICS,LAPTOP PRO,1.0,$1200.00,10.00,2024-01-22,2024-01-10,delivered,Credit Card,United States
3,1004,C004,Diana Prince,diana@shop.com,Clothing,Denim Jeans,2.0,$55.00,0.15,28/01/2024,2024-02-04,DELIVERED,None,GB
4,1005,c005,None,eve@shop.com,food & drink,Organic Coffee,5.0,$12.50,0.00,2024-02-03,2024-02-10,Processing,PayPal,France
5,1006,C006,EVE ADAMS,notanemail,Food & Drink,ORGANIC COFFEE,NaN,$12.50,5.00,2024-02-10,2024-02-18,shipped,Credit Card,FR
6,1007,C007,Frank Castle,frank@shop.com,electronics,Wireless Mouse,2.0,$35.00,0.20,14/02/2024,2024-02-21,Delivered,Debit Card,USA
7,1008,C008,grace hopper,grace@shop.com,CLOTHING,Wool Sweater,1.0,$80.00,0.00,2024-02-20,2024-02-28,SHIPPED,PayPal,United Kingdom
8,1009,c009,HENRY FORD,henry@shop.com,Electronics,USB-C Hub,4.0,$45.00,0.10,2024-03-01,2024-03-09,Delivered,Credit Card,US
9,1010,C010,ida lupino,ida@shop.com,food & drink,Green Tea Pack,6.0,$8.00,0.05,05/03/2024,2024-03-14,Processing,Debit Card,France


In [69]:
# ── PHASE 1: INSPECTION ──────────────────────────────────────────────
# Goal: understand every data quality problem BEFORE writing any fix.
# Rule: never clean what you haven't inspected. Inspection takes 2 minutes.
# Skipping it costs 2 hours of debugging wrong fixes.

print("=" * 55)
print("STEP 1 — SHAPE")
print("=" * 55)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\n" + "=" * 55)
print("STEP 2 — DATA TYPES")
print("=" * 55)
print(df.dtypes)
# ── KEY THING TO NOTICE ──────────────────────────────────────────────
# unit_price shows as "object" (text) instead of float64.
# That is your signal the "$" prefix is blocking numeric parsing.
# Any column that should be numeric but shows as object needs investigation.

print("\n" + "=" * 55)
print("STEP 3 — COLUMNS STORED AS TEXT THAT SHOULD BE NUMERIC")
print("=" * 55)
# select_dtypes returns only columns of the specified dtype
# This is faster than scanning all 14 columns manually
obj_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Object (text) columns: {obj_cols}")
print(f"\nCheck unit_price sample: {df['unit_price'].head(5).tolist()}")

print("\n" + "=" * 55)
print("STEP 4 — NULL COUNT AND PERCENTAGE")
print("=" * 55)
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(1)
null_report = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print(null_report[null_report["null_count"] > 0].to_string())

print("\n" + "=" * 55)
print("STEP 5 — KEY COLUMN VALUE DISTRIBUTIONS")
print("=" * 55)
for col in ["status", "product_category", "country", "payment_method"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

print("\n" + "=" * 55)
print("STEP 6 — NUMERIC SUMMARY")
print("=" * 55)
# NOTE: unit_price will be ABSENT from describe() because it is stored as text
# That confirms the dtype problem we spotted in Step 2
print(df.describe())

print("\n" + "=" * 55)
print("STEP 7 — DUPLICATES")
print("=" * 55)
print(f"Fully identical rows   : {df.duplicated().sum()}")
print(f"Duplicate order_id     : {df.duplicated(subset=['order_id']).sum()}")
print(f"Duplicate customer_id  : {df.duplicated(subset=['customer_id']).sum()}")

STEP 1 — SHAPE
Rows    : 15
Columns : 14

STEP 2 — DATA TYPES
order_id              int64
customer_id          object
customer_name        object
email                object
product_category     object
product_name         object
quantity            float64
unit_price           object
discount_pct        float64
order_date           object
delivery_date        object
status               object
payment_method       object
country              object
dtype: object

STEP 3 — COLUMNS STORED AS TEXT THAT SHOULD BE NUMERIC
Object (text) columns: ['customer_id', 'customer_name', 'email', 'product_category', 'product_name', 'unit_price', 'order_date', 'delivery_date', 'status', 'payment_method', 'country']

Check unit_price sample: ['$1200.00', '$25.00', '$1200.00', '$55.00', '$12.50']

STEP 4 — NULL COUNT AND PERCENTAGE
                null_count  null_%
customer_name            1     6.7
quantity                 1     6.7
unit_price               1     6.7
payment_method           2    13.3

In [70]:
# ── SCHEMA VALIDATION ────────────────────────────────────────────────
# Before cleaning, confirm every expected column exists.
# This catches the case where the client sends a new CSV
# with a renamed or missing column — fails fast with a readable message.

def validate_schema(df, expected_cols):
    """
    Confirm all expected columns are present in the DataFrame.

    Args:
        df            : DataFrame to validate
        expected_cols : list of column name strings that must exist

    Returns:
        True if all columns present

    Raises:
        ValueError with a list of missing columns
    """
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"[SCHEMA ERROR] Missing columns: {missing}\n"
            f"Expected: {expected_cols}\n"
            f"Got     : {df.columns.tolist()}"
        )
    print(f"[SCHEMA] All {len(expected_cols)} expected columns present.")
    return True

try:
    validate_schema(df, EXPECTED_COLS)
except ValueError as e:
    print(e)
    import sys; sys.exit(1)

[SCHEMA] All 14 expected columns present.


In [71]:
# ── PHASE 3: CLEAN — NULLS ───────────────────────────────────────────
# Always work on a copy. Preserve the original for comparison.

df_clean = df.copy()

print("BEFORE cleaning — null counts:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0].to_string())
print()

# ── customer_name: drop rows where name is null ──────────────────────
# A customer order without a name cannot be attributed — drop it
before = len(df_clean)
df_clean = df_clean.dropna(subset=["customer_name"])
print(f"[DROP]   {before - len(df_clean)} rows dropped: null customer_name")

# ── unit_price: fill null with median AFTER fixing the $ prefix ──────
# We flag it here but fix it in the type-conversion cell (Cell 5)
# because fillna on a text column filled with a number causes a type mismatch
# We'll handle it in order: strip $ → convert to float → fillna(median)

# ── payment_method: fill with "Unknown" ──────────────────────────────
df_clean["payment_method"] = df_clean["payment_method"].fillna("Unknown")
print(f"[FILL]   payment_method nulls → 'Unknown'")

# ── quantity: fill null with median ──────────────────────────────────
# Quantity is numeric — median is more robust than mean
median_qty = df_clean["quantity"].median()
df_clean["quantity"] = df_clean["quantity"].fillna(median_qty)
print(f"[FILL]   quantity nulls → median = {median_qty}")

print(f"\n[RESULT] Nulls after this step:")
remaining = df_clean.isnull().sum()
print(remaining[remaining > 0].to_string()
      if remaining.sum() > 0 else "  None — all nulls resolved")

BEFORE cleaning — null counts:
customer_name     1
quantity          1
unit_price        1
payment_method    2

[DROP]   1 rows dropped: null customer_name
[FILL]   payment_method nulls → 'Unknown'
[FILL]   quantity nulls → median = 2.0

[RESULT] Nulls after this step:
unit_price    1


In [100]:
# ── PHASE 3: CLEAN — DATA TYPES ──────────────────────────────────────

print("BEFORE type fixes:")
print(df_clean[["unit_price","discount_pct","order_date","delivery_date"]].dtypes)
print()

# ── FIX 1: Strip "$" from unit_price and convert to float ────────────
# str.replace("$", "") removes the dollar sign character from every value
# str.strip() removes any remaining whitespace around the number
# pd.to_numeric(errors="coerce") converts to float; bad values → NaN
df_clean["unit_price"] = (
    df_clean["unit_price"]
    .astype(str)                        # ensure it is a string first
    .str.replace("$", "", regex=False)  # remove $ — regex=False = literal match
    .str.strip()                        # remove whitespace
    .replace("nan", np.nan)             # "nan" string (from astype(str) on NaN) → real NaN
)
df_clean["unit_price"] = pd.to_numeric(df_clean["unit_price"], errors="coerce")

# Now fill the null that appeared after conversion (was None in original data)
median_price = df_clean["unit_price"].median()
df_clean["unit_price"] = df_clean["unit_price"].fillna(median_price)
print(f"[TYPE]   unit_price: stripped '$', converted to float")
print(f"[FILL]   unit_price nulls → median = ${median_price:.2f}")

# ── FIX 2: discount_pct scale fix ────────────────────────────────────
# Problem: some values are 0–1 (correct: 0.10 = 10%)
#          some values are 0–100 (wrong: 10.0 means 10% but will be used as 1000%)
# Fix: any value > 1 is on the 0–100 scale — divide by 100
wrong_scale_mask = df_clean["discount_pct"] > 1
wrong_count = wrong_scale_mask.sum()
df_clean.loc[wrong_scale_mask, "discount_pct"] = (
    df_clean.loc[wrong_scale_mask, "discount_pct"] / 100
)
print(f"[FIX]    discount_pct: {wrong_count} values divided by 100 (were on 0–100 scale)")

# Confirm all discounts are now between 0 and 1
assert df_clean["discount_pct"].between(0, 1).all(), \
    "[ERROR] discount_pct still has values outside 0–1 range"
print(f"[CHECK]  All discount_pct values now in range [0, 1]")

# ── FIX 3: Parse dates — mixed format handling ────────────────────────
# The dataset has TWO date formats:
#   "2024-01-10"   → YYYY-MM-DD (ISO format — pandas handles this natively)
#   "15/01/2024"   → DD/MM/YYYY (European format — needs dayfirst=True)
# dayfirst=True tells pandas to try DD/MM/YYYY before MM/DD/YYYY
# errors="coerce" turns unparseable values → NaT instead of crashing
for col in ["order_date", "delivery_date"]:
    df_clean[col] = pd.to_datetime(df_clean[col], dayfirst=True, errors="coerce")
    bad = df_clean[col].isnull().sum()
    print(f"[DATE]   {col}: parsed. {bad} values could not be parsed → NaT")

# ── FIX 4: quantity — convert to int after null fill ─────────────────
df_clean["quantity"] = df_clean["quantity"].astype(int)
print(f"[TYPE]   quantity converted to int")

print("\nAFTER type fixes:")
print(df_clean[["unit_price","discount_pct","order_date","delivery_date"]].dtypes)

BEFORE type fixes:
unit_price              float64
discount_pct            float64
order_date       datetime64[ns]
delivery_date    datetime64[ns]
dtype: object

[TYPE]   unit_price: stripped '$', converted to float
[FILL]   unit_price nulls → median = $45.00
[FIX]    discount_pct: 0 values divided by 100 (were on 0–100 scale)
[CHECK]  All discount_pct values now in range [0, 1]
[DATE]   order_date: parsed. 10 values could not be parsed → NaT
[DATE]   delivery_date: parsed. 0 values could not be parsed → NaT
[TYPE]   quantity converted to int

AFTER type fixes:
unit_price              float64
discount_pct            float64
order_date       datetime64[ns]
delivery_date    datetime64[ns]
dtype: object


In [101]:
# ── PHASE 3: CLEAN — TEXT STANDARDISATION ────────────────────────────
# Same 3-step pipeline: .str.strip() → .str.title()
# Applied to every text column that will be used in groupby or displayed

print("BEFORE text cleaning:")
print("status unique         :", df_clean["status"].unique().tolist())
print("product_category unique:", df_clean["product_category"].unique().tolist())
print("country unique        :", df_clean["country"].unique().tolist())

# ── Clean name and product text columns ──────────────────────────────
for col in ["customer_name", "product_name", "product_category", "status", "payment_method"]:
    df_clean[col] = df_clean[col].str.strip().str.title()

# ── customer_id: uppercase for consistency ────────────────────────────
# customer_id is a code (C001) — use .str.upper() not .str.title()
# .str.title() would give "C001" anyway but upper() is more explicit for IDs
df_clean["customer_id"] = df_clean["customer_id"].str.upper()

# ── country: standardise abbreviations to full names ─────────────────
# A mapping dictionary is the correct pattern for value replacement
# df.map() applies the dictionary to every value in the column
# Values not in the dictionary → NaN, so we chain .fillna(df_clean["country"])
# to keep original values that were already correct full names
country_map = {
    "USA"            : "United States",
    "US"             : "United States",
    "UK"             : "United Kingdom",
    "GB"             : "United Kingdom",
    "FR"             : "France",
}
# map() replaces matching keys with their values; non-matches → NaN
# fillna(df_clean["country"]) restores NaN back to the original value
df_clean["country"] = (
    df_clean["country"]
    .map(country_map)
    .fillna(df_clean["country"])
)

print("\nAFTER text cleaning:")
print("status unique         :", df_clean["status"].unique().tolist())
print("product_category unique:", df_clean["product_category"].unique().tolist())
print("country unique        :", df_clean["country"].unique().tolist())

BEFORE text cleaning:
status unique         : ['Delivered', 'Shipped', 'Processing']
product_category unique: ['Electronics', 'Clothing', 'Food & Drink']
country unique        : ['United States', 'United Kingdom', 'France']

AFTER text cleaning:
status unique         : ['Delivered', 'Shipped', 'Processing']
product_category unique: ['Electronics', 'Clothing', 'Food & Drink']
country unique        : ['United States', 'United Kingdom', 'France']


In [102]:
# ── PHASE 3: CLEAN — NEW PROBLEMS ────────────────────────────────────

# ── NEW PROBLEM 1: Invalid email addresses ───────────────────────────
# An email address must contain "@" — that is the minimum validation rule.
# str.contains("@") returns True where "@" is present, False where it isn't.
# We flag invalid emails rather than dropping rows — the rest of the row is valid.

df_clean["email_valid"] = df_clean["email"].str.contains("@", na=False)
# na=False: if email is NaN, treat it as not containing "@" (flag it as invalid)

invalid_emails = df_clean[~df_clean["email_valid"]]
print(f"[EMAIL]  {len(invalid_emails)} invalid email addresses flagged:")
print(invalid_emails[["order_id","customer_name","email","email_valid"]].to_string(index=False))

# ── NEW PROBLEM 2: Negative quantity ─────────────────────────────────
# Quantity physically cannot be negative in an orders dataset.
# (Returns/refunds would be a separate table in a real database.)
# Fix: take the absolute value — a quantity of -3 was likely a data entry error.
neg_qty = (df_clean["quantity"] < 0).sum()
df_clean["quantity"] = df_clean["quantity"].abs()
# .abs() returns the absolute value of every element — negatives become positive
print(f"\n[QTY]    {neg_qty} negative quantities corrected with .abs()")

# ── NEW PROBLEM 3: Delivery date before order date ───────────────────
# Logically impossible: a package cannot be delivered before it was ordered.
# This signals a data entry error — swap the dates or flag the row.
# Strategy: flag and report rather than auto-fix (show the client first)
df_clean["delivery_before_order"] = (
    df_clean["delivery_date"] < df_clean["order_date"]
)
logic_errors = df_clean[df_clean["delivery_before_order"]]
print(f"\n[DATE]   {len(logic_errors)} orders with delivery before order date:")
print(logic_errors[
    ["order_id","customer_name","order_date","delivery_date","delivery_before_order"]
].to_string(index=False))

# For practice: set delivery_date to NaT where the logic violation exists
# In a real project: send these rows back to the client for correction
df_clean.loc[df_clean["delivery_before_order"], "delivery_date"] = pd.NaT
print(f"\n[FIX]    delivery_date set to NaT on {len(logic_errors)} impossible rows")

[EMAIL]  2 invalid email addresses flagged:
 order_id customer_name           email  email_valid
     1003 Charlie Brown charlie_noemail        False
     1006     Eve Adams      notanemail        False

[QTY]    1 negative quantities corrected with .abs()

[DATE]   3 orders with delivery before order date:
 order_id customer_name order_date delivery_date  delivery_before_order
     1001 Alice Johnson 2024-10-01    2024-01-14                   True
     1006     Eve Adams 2024-10-02    2024-02-18                   True
     1014     Maya Gold 2024-05-04    2024-04-14                   True

[FIX]    delivery_date set to NaT on 3 impossible rows


In [103]:
# ── PHASE 3: CLEAN — DUPLICATES ──────────────────────────────────────
print("Duplicate rows by order_id:")
dupes = df_clean[df_clean.duplicated(subset=["order_id"], keep=False)]
print(dupes[["order_id","customer_id","customer_name","product_name"]].to_string(index=False))

before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["order_id"], keep="first")
print(f"\n[DUPES]  Removed {before - len(df_clean)} duplicate orders")
print(f"[RESULT] {len(df_clean)} unique orders remain")

print(f"\nFinal null count after all cleaning:")
nulls = df_clean.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "  Zero nulls remain")
print(f"\n[CLEAN COMPLETE] Shape: {df_clean.shape}")

Duplicate rows by order_id:
 order_id customer_id customer_name   product_name
     1002        C002     Bob Smith Cotton T-Shirt
     1002        C002     Bob Smith Cotton T-Shirt

[DUPES]  Removed 1 duplicate orders
[RESULT] 13 unique orders remain

Final null count after all cleaning:
order_date       9
delivery_date    3

[CLEAN COMPLETE] Shape: (13, 16)


In [104]:
# ── PHASE 4: TRANSFORM — ENGINEER NEW COLUMNS ────────────────────────

# ── gross_revenue = quantity × unit_price (before discount) ──────────
df_clean["gross_revenue"] = df_clean["quantity"] * df_clean["unit_price"]

# ── net_revenue = gross × (1 − discount_pct) ─────────────────────────
# (1 - 0.10) = 0.90 → customer pays 90% of the original price
df_clean["net_revenue"] = (
    df_clean["gross_revenue"] * (1 - df_clean["discount_pct"])
).round(2)

# ── discount_amount = how many dollars were discounted ───────────────
df_clean["discount_amount"] = (
    df_clean["gross_revenue"] - df_clean["net_revenue"]
).round(2)

# ── delivery_days: number of days between order and delivery ─────────
# Subtracting two datetime columns produces a timedelta Series
# .dt.days extracts the integer number of days from each timedelta value
df_clean["delivery_days"] = (
    df_clean["delivery_date"] - df_clean["order_date"]
).dt.days
# Rows with NaT delivery_date will produce NaN delivery_days — expected

# ── is_late: flag orders taking more than 7 days ─────────────────────
# A business rule — adjust 7 to whatever the client's SLA actually is
df_clean["is_late"] = df_clean["delivery_days"] > 7

# ── Extract date parts for trend grouping ────────────────────────────
df_clean["order_year"]    = df_clean["order_date"].dt.year
df_clean["order_month"]   = df_clean["order_date"].dt.month
df_clean["order_month_name"] = df_clean["order_date"].dt.strftime("%B")

print("[TRANSFORM] New columns added:")
print(df_clean[[
    "order_id","product_name","gross_revenue","net_revenue",
    "discount_amount","delivery_days","is_late"
]].to_string(index=False))

print(f"\n[CHECK] Zero new nulls in revenue columns: "
      f"{df_clean[['gross_revenue','net_revenue']].isnull().sum().sum() == 0}")

[TRANSFORM] New columns added:
 order_id      product_name  gross_revenue  net_revenue  discount_amount  delivery_days  is_late
     1001        Laptop Pro         1200.0      1080.00           120.00            NaN    False
     1002    Cotton T-Shirt           75.0        71.25             3.75            NaN    False
     1003        Laptop Pro         1200.0      1080.00           120.00            NaN    False
     1004       Denim Jeans          110.0        93.50            16.50            NaN    False
     1006    Organic Coffee           25.0        23.75             1.25            NaN    False
     1007    Wireless Mouse           70.0        56.00            14.00            NaN    False
     1008      Wool Sweater           80.0        80.00             0.00            NaN    False
     1009         Usb-C Hub          180.0       162.00            18.00           66.0     True
     1010    Green Tea Pack           48.0        45.60             2.40            NaN    False

In [105]:
# ── PHASE 5: ANALYSIS — KPI SUMMARIES ────────────────────────────────

# ── KPI 1: Summary by country ────────────────────────────────────────
country_summary = (
    df_clean
    .groupby("country", as_index=False)
    .agg(
        gross_revenue     = ("gross_revenue",  "sum"),
        net_revenue       = ("net_revenue",    "sum"),
        discount_given    = ("discount_amount","sum"),
        order_count       = ("order_id",       "count"),
        avg_delivery_days = ("delivery_days",  "mean"),
        late_orders       = ("is_late",        "sum")   # True=1, False=0 → sum = count
    )
    .round(2)
)
country_summary["late_rate_pct"] = (
    country_summary["late_orders"] / country_summary["order_count"] * 100
).round(1)
country_summary["revenue_share_pct"] = (
    country_summary["net_revenue"] / country_summary["net_revenue"].sum() * 100
).round(1)
country_summary = country_summary.sort_values("net_revenue", ascending=False)
print("=== COUNTRY SUMMARY ===")
print(country_summary.to_string(index=False))

# ── KPI 2: Summary by product category ───────────────────────────────
category_summary = (
    df_clean
    .groupby("product_category", as_index=False)
    .agg(
        net_revenue       = ("net_revenue",    "sum"),
        gross_revenue     = ("gross_revenue",  "sum"),
        order_count       = ("order_id",       "count"),
        avg_discount_pct  = ("discount_pct",   "mean"),
        units_sold        = ("quantity",       "sum")
    )
    .round(2)
)
category_summary = category_summary.sort_values("net_revenue", ascending=False)
print("\n=== CATEGORY SUMMARY ===")
print(category_summary.to_string(index=False))

# ── KPI 3: Top 5 products by net revenue ─────────────────────────────
product_summary = (
    df_clean
    .groupby("product_name", as_index=False)
    .agg(
        net_revenue  = ("net_revenue", "sum"),
        units_sold   = ("quantity",    "sum"),
        order_count  = ("order_id",    "count")
    )
    .round(2)
    .sort_values("net_revenue", ascending=False)
)
top5_products = product_summary.head(5)
print("\n=== TOP 5 PRODUCTS ===")
print(top5_products.to_string(index=False))

# ── KPI 4: Monthly trend ──────────────────────────────────────────────
monthly = (
    df_clean
    .groupby(["order_year","order_month","order_month_name"], as_index=False)
    .agg(
        net_revenue  = ("net_revenue", "sum"),
        order_count  = ("order_id",    "count")
    )
    .sort_values(["order_year","order_month"])
    .reset_index(drop=True)
)
print("\n=== MONTHLY TREND ===")
print(monthly[["order_month_name","net_revenue","order_count"]].to_string(index=False))

=== COUNTRY SUMMARY ===
       country  gross_revenue  net_revenue  discount_given  order_count  avg_delivery_days  late_orders  late_rate_pct  revenue_share_pct
 United States         2885.0      2593.30          291.70            6               66.0            1           16.7               85.8
United Kingdom          395.0       348.75           46.25            4                NaN            0            0.0               11.5
        France           83.5        79.85            3.65            3                NaN            0            0.0                2.6

=== CATEGORY SUMMARY ===
product_category  net_revenue  gross_revenue  order_count  avg_discount_pct  units_sold
     Electronics      2593.30         2885.0            6              0.11          11
        Clothing       348.75          395.0            4              0.10           8
    Food & Drink        79.85           83.5            3              0.03          11

=== TOP 5 PRODUCTS ===
     product_name  net

In [106]:
# ── PHASE 5: SORT AND RANK ────────────────────────────────────────────

# ── Rank products by net_revenue ─────────────────────────────────────
df_clean["revenue_rank"] = (
    df_clean["net_revenue"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

# ── Multi-column sort: country A→Z, then net_revenue high→low ────────
print("=== ORDERS SORTED BY COUNTRY THEN REVENUE ===")
df_sorted = df_clean.sort_values(
    by=["country", "net_revenue"],
    ascending=[True, False]
).reset_index(drop=True)
print(df_sorted[[
    "order_id","customer_name","country","product_name","net_revenue","revenue_rank"
]].to_string(index=False))

# ── Top 3 and bottom 3 orders by net revenue ─────────────────────────
print("\n=== TOP 3 ORDERS ===")
print(df_clean.nlargest(3,"net_revenue")[
    ["order_id","customer_name","product_name","net_revenue","revenue_rank"]
].to_string(index=False))

print("\n=== BOTTOM 3 ORDERS ===")
print(df_clean.nsmallest(3,"net_revenue")[
    ["order_id","customer_name","product_name","net_revenue","revenue_rank"]
].to_string(index=False))

# ── Best order per country ────────────────────────────────────────────
print("\n=== BEST ORDER PER COUNTRY ===")
best_per_country = (
    df_clean
    .sort_values("net_revenue", ascending=False)
    .groupby("country")
    .first()
    .reset_index()
)[["country","customer_name","product_name","net_revenue"]]
print(best_per_country.to_string(index=False))

=== ORDERS SORTED BY COUNTRY THEN REVENUE ===
 order_id customer_name        country      product_name  net_revenue  revenue_rank
     1010    Ida Lupino         France    Green Tea Pack        45.60             9
     1006     Eve Adams         France    Organic Coffee        23.75            11
     1013     Luke Cage         France   Sparkling Water        10.50            12
     1011     Jack Ryan United Kingdom     Slim Trousers       104.00             4
     1004  Diana Prince United Kingdom       Denim Jeans        93.50             5
     1008  Grace Hopper United Kingdom      Wool Sweater        80.00             6
     1002     Bob Smith United Kingdom    Cotton T-Shirt        71.25             7
     1001 Alice Johnson  United States        Laptop Pro      1080.00             1
     1003 Charlie Brown  United States        Laptop Pro      1080.00             1
     1014     Maya Gold  United States Bluetooth Speaker       174.80             2
     1009    Henry Ford  Unite

In [108]:
# ── PHASE 6: CHARTS ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from io import BytesIO

def make_chart_buffer(fig, dpi=150):
    """Save a matplotlib figure to a BytesIO PNG buffer."""
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    buf.seek(0)   # rewind to start — critical before openpyxl reads it
    plt.close(fig)
    return buf

COLORS = ["#1B5E20","#2E7D32","#388E3C","#43A047","#66BB6A"]

# ── Chart 1: Net revenue by country (vertical bar) ───────────────────
fig1, ax1 = plt.subplots(figsize=(10,5))
bars = ax1.bar(
    country_summary["country"],
    country_summary["net_revenue"],
    color=COLORS[:len(country_summary)],
    edgecolor="white", width=0.55
)
for b in bars:
    h = b.get_height()
    ax1.text(b.get_x()+b.get_width()/2, h+20, f"${h:,.0f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold", color="#1B5E20")
ax1.set_title("Net Revenue by Country", fontsize=14, fontweight="bold",
              pad=15, color="#1B5E20")
ax1.set_xlabel("Country"); ax1.set_ylabel("Net Revenue ($)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
ax1.yaxis.grid(True, linestyle="--", alpha=0.4); ax1.set_axisbelow(True)
plt.tight_layout()
buf_chart1 = make_chart_buffer(fig1)
print("[CHART 1] Net revenue by country — done")

# ── Chart 2: Top 5 products by net revenue (horizontal bar) ──────────
fig2, ax2 = plt.subplots(figsize=(10,4))
prod_plot = top5_products.sort_values("net_revenue", ascending=True)
hbars = ax2.barh(
    prod_plot["product_name"], prod_plot["net_revenue"],
    color="#2E7D32", edgecolor="white", height=0.55
)
for b in hbars:
    w = b.get_width()
    ax2.text(w+20, b.get_y()+b.get_height()/2, f"${w:,.0f}",
             ha="left", va="center", fontsize=10, fontweight="bold", color="#1B5E20")
ax2.set_title("Top 5 Products by Net Revenue", fontsize=14, fontweight="bold",
              pad=15, color="#1B5E20")
ax2.set_xlabel("Net Revenue ($)")
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
ax2.xaxis.grid(True, linestyle="--", alpha=0.4); ax2.set_axisbelow(True)
plt.tight_layout()
buf_chart2 = make_chart_buffer(fig2)
print("[CHART 2] Top 5 products — done")

# ── Chart 3: Monthly net revenue trend (line) ─────────────────────────
fig3, ax3 = plt.subplots(figsize=(10,4))
ax3.plot(monthly["order_month_name"], monthly["net_revenue"],
         color="#1B5E20", marker="o", linewidth=2.5, markersize=8,
         markerfacecolor="white", markeredgecolor="#1B5E20", markeredgewidth=2)
ax3.fill_between(monthly["order_month_name"], monthly["net_revenue"],
                 alpha=0.12, color="#1B5E20")
for x, y in zip(monthly["order_month_name"], monthly["net_revenue"]):
    ax3.annotate(f"${y:,.0f}", xy=(x,y), xytext=(0,12),
                 textcoords="offset points", ha="center",
                 fontsize=9, color="#1B5E20", fontweight="bold")
ax3.set_title("Monthly Net Revenue Trend", fontsize=14, fontweight="bold",
              pad=15, color="#1B5E20")
ax3.set_xlabel("Month"); ax3.set_ylabel("Net Revenue ($)")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
ax3.yaxis.grid(True, linestyle="--", alpha=0.4); ax3.set_axisbelow(True)
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
buf_chart3 = make_chart_buffer(fig3)
print("[CHART 3] Monthly trend — done")

[CHART 1] Net revenue by country — done
[CHART 2] Top 5 products — done
[CHART 3] Monthly trend — done


In [109]:
# ── PHASE 6: EXCEL EXPORT ────────────────────────────────────────────
import openpyxl
import openpyxl.drawing.image
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils  import get_column_letter

def format_worksheet(ws, header_color="1B5E20", currency_cols=None):
    """Apply professional formatting to any worksheet."""
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.fill      = PatternFill("solid", fgColor=header_color)
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for col in ws.columns:
        max_len    = max(len(str(c.value or "")) for c in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max_len + 4
    ws.freeze_panes = "A2"
    if currency_cols:
        for ci in currency_cols:
            for row in ws.iter_rows(min_row=2, min_col=ci, max_col=ci):
                for cell in row:
                    cell.number_format = "#,##0.00"
    thin = Side(style="thin")
    for cell in ws[1]:
        cell.border = Border(bottom=thin)
    grey = PatternFill("solid", fgColor="F2F2F2")
    for ri, row in enumerate(ws.iter_rows(min_row=2), start=2):
        if ri % 2 == 0:
            for cell in row:
                cell.fill = grey


def add_charts_sheet(writer, b1, b2, b3):
    """Embed 3 chart images on a dedicated Charts sheet."""
    wb = writer.book
    ws = wb.create_sheet("Charts")
    ws.sheet_properties.tabColor = "1B5E20"
    ws.sheet_view.showGridLines   = False
    ws.column_dimensions["A"].width = 2
    ws["B1"] = "E-Commerce Performance Charts"
    ws["B1"].font = Font(bold=True, size=14, color="1B5E20")
    IMG_W, IMG_H  = 600, 320
    rows_per_chart = int(IMG_H / 15) + 2
    for i, (buf, anchor) in enumerate(
        [(b1,"B3"), (b2,f"B{3+rows_per_chart}"), (b3,f"B{3+rows_per_chart*2}")]
    ):
        img        = openpyxl.drawing.image.Image(buf)
        img.width  = IMG_W
        img.height = IMG_H
        ws.add_image(img, anchor)
        print(f"    [CHART {i+1}] anchored at {anchor}")


OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
timestamp   = datetime.now().strftime("%Y_%m_%d_%H%M")
output_path = OUTPUT_FOLDER / f"ecommerce_report_{timestamp}.xlsx"

try:
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        print("[EXPORT] Writing sheets...")

        country_summary.to_excel(writer, sheet_name="Country Summary",  index=False)
        format_worksheet(writer.sheets["Country Summary"],  currency_cols=[2,3,4])

        category_summary.to_excel(writer, sheet_name="Category Summary", index=False)
        format_worksheet(writer.sheets["Category Summary"], currency_cols=[2,3])

        top5_products.to_excel(writer, sheet_name="Top 5 Products", index=False)
        format_worksheet(writer.sheets["Top 5 Products"],   currency_cols=[2])

        print("[EXPORT] Embedding charts...")
        add_charts_sheet(writer, buf_chart1, buf_chart2, buf_chart3)

        wb = writer.book
        wb.move_sheet("Charts", offset=-len(wb.sheetnames)+1)

    print(f"\n[DONE] Report saved → {output_path}")

except Exception as e:
    print(f"[ERROR] Export failed: {e}")
    raise

[EXPORT] Writing sheets...
[EXPORT] Embedding charts...
    [CHART 1] anchored at B3
    [CHART 2] anchored at B26
    [CHART 3] anchored at B49

[DONE] Report saved → output\ecommerce_report_2026_06_12_2326.xlsx


In [110]:
# ── PHASE 7: EMAIL DELIVERY ───────────────────────────────────────────
# Reusing the exact same send functions from the sales project.
# Only the email body content changes — the machinery is identical.

from dotenv import load_dotenv
import os, smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text      import MIMEText
from email.mime.base      import MIMEBase
from email                import encoders

load_dotenv()
SENDER_EMAIL    = os.getenv("SENDER_EMAIL")
GMAIL_PASSWORD  = os.getenv("GMAIL_APP_PASSWORD")
RECIPIENT_EMAIL = os.getenv("RECIPIENT_EMAIL")
RECIPIENT_NAME  = os.getenv("RECIPIENT_NAME", "Team")


def build_ecommerce_email(country_df, category_df, filename):
    """Build HTML email body with e-commerce KPI highlights."""
    total_net    = country_df["net_revenue"].sum()
    total_disc   = country_df["discount_given"].sum()
    total_orders = country_df["order_count"].sum()
    top_country  = country_df.iloc[0]
    report_date  = datetime.now().strftime("%B %d, %Y")

    rows = ""
    for _, r in country_df.iterrows():
        bg = "#E8F5E9" if _ % 2 == 0 else "#FFFFFF"
        rows += f"""
        <tr style="background:{bg}">
          <td style="padding:8px 12px">{r['country']}</td>
          <td style="padding:8px 12px;text-align:right">${r['net_revenue']:,.2f}</td>
          <td style="padding:8px 12px;text-align:right">{int(r['order_count'])}</td>
          <td style="padding:8px 12px;text-align:right">${r['avg_delivery_days']:.1f}d</td>
          <td style="padding:8px 12px;text-align:right">{r['late_rate_pct']}%</td>
        </tr>"""

    return f"""<html><body style="font-family:Arial,sans-serif;color:#333;max-width:700px;margin:0 auto;padding:20px">
      <div style="background:#1B5E20;padding:20px 24px;border-radius:6px 6px 0 0">
        <h2 style="color:#fff;margin:0;font-size:20px">Weekly E-Commerce Report</h2>
        <p style="color:#A5D6A7;margin:6px 0 0;font-size:13px">Generated automatically — {report_date}</p>
      </div>
      <div style="background:#F9FBF9;padding:20px 24px;border:1px solid #C8E6C9;border-top:none">
        <p style="margin:0 0 12px">Hi {RECIPIENT_NAME},</p>
        <p style="margin:0">Your weekly e-commerce report is attached. Key highlights below.</p>
      </div>
      <div style="display:flex;gap:12px;padding:20px 24px 0;background:#F9FBF9;border:1px solid #C8E6C9;border-top:none">
        <div style="flex:1;background:#fff;border:1px solid #C8E6C9;border-radius:6px;padding:14px;text-align:center">
          <div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.5px">Net Revenue</div>
          <div style="font-size:22px;font-weight:700;color:#1B5E20;margin-top:4px">${total_net:,.2f}</div>
        </div>
        <div style="flex:1;background:#fff;border:1px solid #C8E6C9;border-radius:6px;padding:14px;text-align:center">
          <div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.5px">Total Orders</div>
          <div style="font-size:22px;font-weight:700;color:#1B5E20;margin-top:4px">{int(total_orders)}</div>
        </div>
        <div style="flex:1;background:#fff;border:1px solid #C8E6C9;border-radius:6px;padding:14px;text-align:center">
          <div style="font-size:11px;color:#888;text-transform:uppercase;letter-spacing:.5px">Discount Given</div>
          <div style="font-size:22px;font-weight:700;color:#1B5E20;margin-top:4px">${total_disc:,.2f}</div>
        </div>
      </div>
      <div style="padding:20px 24px;background:#F9FBF9;border:1px solid #C8E6C9;border-top:none">
        <h3 style="margin:0 0 12px;color:#1B5E20;font-size:15px">Revenue by Country</h3>
        <table style="width:100%;border-collapse:collapse;font-size:13px;border:1px solid #C8E6C9">
          <thead><tr style="background:#1B5E20;color:#fff">
            <th style="padding:8px 12px;text-align:left">Country</th>
            <th style="padding:8px 12px;text-align:right">Net Revenue</th>
            <th style="padding:8px 12px;text-align:right">Orders</th>
            <th style="padding:8px 12px;text-align:right">Avg Delivery</th>
            <th style="padding:8px 12px;text-align:right">Late %</th>
          </tr></thead>
          <tbody>{rows}</tbody>
        </table>
      </div>
      <div style="padding:14px 24px;background:#ECEFF1;border:1px solid #C8E6C9;border-top:none;border-radius:0 0 6px 6px">
        <p style="margin:0;font-size:11px;color:#888">Auto-generated by Python. Attached: <strong>{filename}</strong></p>
      </div>
    </body></html>"""


# ── Send the report (set send_email=True once credentials are in .env) ──
send_email = False   # ← change to True for live delivery

if send_email and SENDER_EMAIL and GMAIL_PASSWORD:
    import yagmail
    body  = build_ecommerce_email(country_summary, category_summary, output_path.name)
    subj  = f"Weekly E-Commerce Report — {datetime.now().strftime('%B %d, %Y')}"
    yag   = yagmail.SMTP(SENDER_EMAIL, GMAIL_PASSWORD)
    yag.send(to=RECIPIENT_EMAIL, subject=subj,
             contents=[body, str(output_path)])
    yag.close()
    print(f"[EMAIL] Report sent to {RECIPIENT_EMAIL}")
else:
    body = build_ecommerce_email(country_summary, category_summary, output_path.name)
    from IPython.display import HTML
    print("[PREVIEW] Email body preview (send_email=False):")
    display(HTML(body))

[PREVIEW] Email body preview (send_email=False):


Country,Net Revenue,Orders,Avg Delivery,Late %
United States,"$2,593.30",6,$66.0d,16.7%
United Kingdom,$348.75,4,$nand,0.0%
France,$79.85,3,$nand,0.0%


In [111]:
# ── FINAL NARRATIVE SUMMARY ───────────────────────────────────────────
print("=" * 58)
print("  E-COMMERCE REPORT — KEY FINDINGS")
print("=" * 58)

top_c    = country_summary.iloc[0]
top_cat  = category_summary.iloc[0]
top_prod = top5_products.iloc[0]
late_pct = (df_clean["is_late"].sum() / len(df_clean) * 100).round(1)
inv_mail = (~df_clean["email_valid"]).sum()

print(f"""
1. TOP COUNTRY   : {top_c['country']} — ${top_c['net_revenue']:,.2f} net revenue
                   ({top_c['revenue_share_pct']}% of total)

2. TOP CATEGORY  : {top_cat['product_category']} — ${top_cat['net_revenue']:,.2f}
                   across {int(top_cat['order_count'])} orders

3. TOP PRODUCT   : {top_prod['product_name']} — ${top_prod['net_revenue']:,.2f}

4. LATE ORDERS   : {late_pct}% of orders exceeded the 7-day delivery SLA
                   — review logistics for {country_summary.loc[
                       country_summary['late_rate_pct'].idxmax(), 'country']}

5. DATA QUALITY  : {inv_mail} invalid email addresses flagged for correction
                   1 impossible delivery date corrected (delivered before ordered)
                   All discount percentages normalised to 0–1 scale
""")
print("=" * 58)
print(f"  File : {output_path.name}")
print(f"  Path : {output_path.parent.resolve()}")
print("=" * 58)

  E-COMMERCE REPORT — KEY FINDINGS

1. TOP COUNTRY   : United States — $2,593.30 net revenue
                   (85.8% of total)

2. TOP CATEGORY  : Electronics — $2,593.30
                   across 6 orders

3. TOP PRODUCT   : Laptop Pro — $2,160.00

4. LATE ORDERS   : 7.7% of orders exceeded the 7-day delivery SLA
                   — review logistics for United States

5. DATA QUALITY  : 2 invalid email addresses flagged for correction
                   1 impossible delivery date corrected (delivered before ordered)
                   All discount percentages normalised to 0–1 scale

  File : ecommerce_report_2026_06_12_2326.xlsx
  Path : C:\Users\aman9\OneDrive\Desktop\My code\output
